# Solar Filament Segmentation Challenge 2026: full pipeline

This notebook reproduces `submission_public_0.38.csv` (public PQ 0.38) from the released weights, without training. Pipeline: 12 crop-level ResNet34 U-Nets + 3 whole-disk ConvNeXt-Tiny U-Nets, flip TTA, averaged probability map, candidate forest at six thresholds, LightGBM scoring, exact antichain selection. See `README.md` and the report for details.

A GPU is needed for the first two steps (about 45 minutes on an RTX 5060). Set `DATA` to the competition folder.

In [ ]:
import os, subprocess, sys, urllib.request
from pathlib import Path

DATA = os.environ.get('FILAMENT_DATA', 'data/kagglehub/competitions/filament-segmentation-2026')  # folder with test/ (Kaggle: /kaggle/input/filament-segmentation-2026)
RELEASE_URL = 'https://github.com/i-oeng/solar/releases/download/v1.0' 
WEIGHTS = Path('weights')
print(sys.version)

## 1. Requirements

In [ ]:
!{sys.executable} -m pip install -q -r requirements.txt

## 2. Weights

The 15 model files and the scorer are release assets (too large for plain git). They are downloaded into `weights/` unless already present.

In [ ]:
NAMES = ['rescorer_tree_a.txt'] + [f'convnext/f{k}_1536.pt' for k in range(3)] + [
    f'resnet34/annotation_round_f{f}_s{s}_{v}.pt' for f in range(3) for s in (2026, 2029) for v in ('most_complete', 'uniform')]
for name in NAMES:
    target = WEIGHTS / name
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f'{RELEASE_URL}/{name.replace("/", "__")}', target)
print(len(NAMES), 'files ready')

## 3. Predict

`reproduce.py` runs the whole chain (ResNet maps, ConvNeXt maps, blend, forest selection, validation). Add `--limit 5` for a quick check on the first five test images.

In [ ]:
!{sys.executable} reproduce.py --data {DATA} --weights weights --work work

## 4. Compare with the submitted file

In [ ]:
import hashlib
a, b = Path('work/rescored.csv'), Path('submission_public_0.38.csv')
h = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
print('identical' if h(a) == h(b) else 'differs (GPU kernels can change last-digit probabilities); compare row counts below')
print(sum(1 for _ in a.open()) - 1, 'rows reproduced,', sum(1 for _ in b.open()) - 1, 'rows submitted')

## 5. Example figure

In [ ]:
!{sys.executable} scripts/make_report_figures.py work/rescored.csv work/figures work/maps_blend
from IPython.display import Image
Image('work/figures/examples.png')